This notebook was generated using AI under general human direction. At the time of generation, the contents have not been comprehensively reviewed by a human analyst.

# SVM with a polynomial kernel

This notebook uses a support vector classifier (SVC) to predict diabetes from glucose alone. It then shows that the polynomial kernel is equivalent to manually adding a squared dimension and fitting a linear classifier.

----

## Setup

### Load packages

`pandas` and `numpy` handle data and numeric work, `matplotlib` and `seaborn` handle plotting, and `scikit-learn` provides the SVC, train/test splitting, scaling, and accuracy.

In [ ]:
# load-packages
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

### Load data

We read the data and look at the first rows to confirm it loaded correctly and to see which columns exist (`bloodpressure`, `glucose`, `diabetes`).

In [ ]:
# load-data
df = pd.read_csv("data_diabetes.csv")
df.head()

## Explore the data

### Plot both features

Plotting both features shows how well each separates people with and without diabetes. Color and marker style both encode `diabetes`, and semi-transparent points (`alpha=0.5`) make overlap visible.

In [ ]:
# plot-both-features
sns.scatterplot(df,
        x="bloodpressure",
        y="glucose",
        style=df["diabetes"],
        hue=df["diabetes"],
        s=150,
        alpha=0.5,
        palette=["orange", "blue"])

### Plot glucose alone

In order to plot glucose alone, we'll set `x="glucose"` and `y=0`. This puts every point on a single number line and shows how well glucose by itself separates the two groups. If the groups overlap, a simple threshold would not classify them well, which motivates using an SVM.

In [ ]:
# plot-glucose-only
sns.scatterplot(df,
        x="glucose",
        y=0,
        style=df["diabetes"],
        hue=df["diabetes"],
        s=150,
        alpha=0.5,
        palette=["orange", "blue"])

### Check for missing or non-numeric data

SVMs need numeric input with no missing values. The data types should all be numeric (stray characters would show up as `object` columns), and the null counts should all be zero.

In [ ]:
# check-data-quality
print(df.dtypes)
df.isnull().sum()

## Prepare the data

### Define features and target

We will use `glucose` to predict `diabetes`, so we drop `bloodpressure`. Using a single feature keeps the polynomial kernel easy to visualize.

`X` holds the features. `y` is deliberately a 1D Series, because `SVC` expects a 1D target. Using `df[['diabetes']]` would give a DataFrame instead.

In [ ]:
# define-features-and-target
X = df.drop(["diabetes", "bloodpressure"], axis=1)
X.head()

In [ ]:
y = df["diabetes"]
y[:5] # print the first 5 values

### Split into training and testing sets

We fit the model on 80% of the data and evaluate it on the 20% it has never seen. This gives an honest estimate of performance on new data. `random_state` makes the split reproducible.

In [ ]:
# train-test-split
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42)

### Standardize the features

SVMs are sensitive to feature scale, so we standardize each feature to have a mean of 0 and a standard deviation of 1 (subtract the mean, divide by the standard deviation).

To avoid data leakage, we calculate the mean and standard deviation from the training data only, then use those values to standardize both the training and testing data:

- `fit_transform()` learns the values from the training data and applies them.
- `transform()` applies the same training values to the test data.

`set_output(transform="pandas")` keeps the results as DataFrames so column names are preserved.

In [ ]:
# standardize-features
scaler = StandardScaler().set_output(transform="pandas")
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Plot the scaled test data

We look at the test set on a single axis to see how well blood pressure alone separates the groups. Any remaining overlap means the test data still needs a classifier more flexible than a single threshold.

In [ ]:
# plot-scaled-test-data
sns.scatterplot(X_test_scaled,
        x="glucose",
        y=0,
        style=y_test,
        hue=y_test,
        s=150,
        alpha=0.5,
        palette=["orange", "blue"])

## Polynomial kernel SVM

### Fit the model

The polynomial kernel lets the SVM find a boundary in a higher-dimensional space without computing that space explicitly. Here `degree=2` and `coef0=0.5` match the parameters from the StatQuest video **[Support Vector Machines Part 2: The Polynomial Kernel](https://youtu.be/Toet3EiSFcM)**.

In [ ]:
# fit-poly-model
poly_model = SVC(kernel="poly", degree=2, coef0=0.5)
poly_model.fit(X_train_scaled, y_train)

### Evaluate the model

We run the test data through the model and build a confusion matrix, which shows the kinds of errors it makes (false positives vs. false negatives). The accuracy score gives a single summary number. Both are computed on the test set only.

In [ ]:
# evaluate-poly-model
y_pred = poly_model.predict(X_test_scaled)

poly_confusion_matrix = pd.crosstab(y_test,
                                    y_pred,
                                    rownames=["Actual"],
                                    colnames=["Predicted"])
print(poly_confusion_matrix)

poly_accuracy = accuracy_score(y_pred, y_test)
print(f"Accuracy: {poly_accuracy:.2f}")

And we see that the model does a pretty good job predicting who has diabetes.

# BAM!

## Visualize the kernel by hand

The polynomial kernel never actually builds the higher-dimensional data. It uses the **kernel trick** to compute relationships between points directly, so there are no transformed coordinates to plot and no classifier line to draw.

To visualize things, we do the transformation ourselves, then fit a *linear* classifier in the new space.

### Add a squared dimension

We copy the scaled data and add a new column, `glucose^2`. This turns the 1D data into 2D data, where the groups can be separated by a straight line. We apply the same transformation to the training and testing data.

In [ ]:
# add-squared-dimension
X_linear_train = X_train_scaled.copy()
X_linear_train["glucose^2"] = X_train_scaled ** 2
X_linear_train.head()

In [ ]:
X_linear_test = X_test_scaled.copy()
X_linear_test["glucose^2"] = X_test_scaled ** 2
X_linear_test.head()

### Fit a linear SVM on the transformed training data

Because the data are already in the higher dimension, the `linear` kernel can find the separating line directly without any further transformation.

In [ ]:
# fit-linear-model
linear_model = SVC(kernel="linear")
linear_model.fit(X_linear_train, y_train)

### Now run the testing data through the new model

In [ ]:
y_pred_linear = linear_model.predict(X_linear_test)

### Compare the two models

If the polynomial kernel is equivalent to the manual transformation plus a linear classifier, the two confusion matrices and the two accuracy scores should match.

In [ ]:
# compare-models
linear_confusion_matrix = pd.crosstab(y_test,
                                      y_pred_linear,
                                      rownames=["Actual"],
                                      colnames=["Predicted"])
print(linear_confusion_matrix)
print(poly_confusion_matrix)

linear_accuracy = accuracy_score(y_pred_linear, y_test)
print(f"Linear accuracy: {linear_accuracy:.2f}")
print(f"Polynomial accuracy: {poly_accuracy:.2f}")

And we see that both models match.

# DOUBLE BAM!!

## Draw the classifier

### Convert coefficients to a slope and intercept

scikit-learn returns the line as coefficients $w_0$, $w_1$ and $b$ in the equation...

$$w_0 x + w_1 y + b = 0$$

To draw the line we need slope-intercept form. Solving for $y$ gives us...

$$y = -\frac{w_0}{w_1}x - \frac{b}{w_1}$$

...so the slope is $-w_0/w_1$ and the intercept is $-b/w_1$.

In [ ]:
# get the coefficients for the classifier
w = linear_model.coef_[0]
b = linear_model.intercept_[0]

# solve for the slope and intercept
slope = -w[0] / w[1]
y_intercept = -b / w[1]
print(slope)
print(y_intercept)

### Build the line coordinates

These arrays hold evenly spaced x values and the matching y values on the classifier line. The final plot draws the line with `ax.axline`, which only needs a point and a slope, so these arrays are not used in the plot.

In [ ]:
# build-line-coordinates
classifier_x = np.linspace(X_linear_test.min(), X_linear_test.max(), 100)
classifier_y = slope * classifier_x + y_intercept

### Plot the transformed test data and the classifier

With blood pressure squared as a second axis, a straight line (red) should separate the two groups. Points on either side of the line are classified accordingly, which is the same classification the polynomial-kernel SVM makes in the original 1D space.

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(data=X_linear_test, 
        x="glucose", 
        y="glucose^2",
        style=y_test,
        hue=y_test,         
        s=150,
        alpha=0.4,
        palette=["orange", "blue"],
        ax=ax)
ax.axline((0, y_intercept), 
            slope=slope, 
            color='red', 
            label='classifier')

# TRIPLE BAM!!!